## DBTC-1065 

### Steps
* Re-run DExposure from 2024-09-30

##RBTC-131

In [0]:
env="p"

In [0]:

spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryactive.casecode
ADD COLUMN CommitteeDecisionTypeCode_new STRING AFTER CommitteeDecisionTypeCode;
""")

spark.sql(f"""
UPDATE `{env}-cdh`.primaryactive.casecode
SET CommitteeDecisionTypeCode_new = cast(CommitteeDecisionTypeCode as string);
""")

spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryactive.casecode
DROP COLUMN CommitteeDecisionTypeCode;
""")

spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryactive.casecode
RENAME COLUMN CommitteeDecisionTypeCode_new TO CommitteeDecisionTypeCode;
""")

In [0]:
spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryarchive.casecode
ADD COLUMN CommitteeDecisionTypeCode_new STRING AFTER CommitteeDecisionTypeCode;
""")

spark.sql(f"""
UPDATE `{env}-cdh`.primaryarchive.casecode
SET CommitteeDecisionTypeCode_new = cast(CommitteeDecisionTypeCode as string);
""")


spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryarchive.casecode
DROP COLUMN CommitteeDecisionTypeCode;
""")


spark.sql(f"""
ALTER TABLE `{env}-cdh`.primaryarchive.casecode
RENAME COLUMN CommitteeDecisionTypeCode_new TO CommitteeDecisionTypeCode;
""")

In [0]:
spark.sql(f"""
MERGE INTO `{env}-cdh`.primaryactive.casecode cc
USING `{env}-cdh`.landed.qc_case_master cm
ON cm.case_code = cc.CaseCodeKey
and cm.BusinessDate=cc.BusinessDate
and cc.sourceSystem='qc'
WHEN MATCHED THEN
UPDATE SET
cc.CommitteeDecisionTypeCode = cast(cm.CAEX_CUSTOM_TEXT_49 as string)
""")

##DBTC-1062

In [0]:
%sql
CREATE TABLE `d-cdh`.landing.qc_case_assignment_test_d AS
SELECT DISTINCT *
FROM `p-cdh`.landing.qc_case_assignment
WHERE  Cluster_0='2024-03-20';

MERGE INTO `d-cdh`.landing.qc_case_assignment_test_d c
USING (
    SELECT case_code, assignment_process, extc_code
    FROM (
        SELECT row_number() OVER (
            PARTITION BY case_code 
            ORDER BY 
                CASE WHEN assignment_process = 'Unhandled' THEN 0 ELSE 1 END DESC, 
                CASE WHEN extc_code IS NULL THEN 0 ELSE 1 END DESC
        ) AS row_num,
        case_code, assignment_process, extc_code
        FROM `d-cdh`.landing.qc_case_assignment_test_d
        WHERE Cluster_0='2024-03-20'
    ) AS a
    WHERE a.row_num = 2
) b
ON c.case_code = b.case_code 
   AND c.assignment_process = b.assignment_process 
   AND COALESCE(c.extc_code, 1) = COALESCE(b.extc_code, 1)
   AND c.Cluster_0='2024-03-20'
WHEN MATCHED THEN
    DELETE;

DELETE from `p-cdh`.landing.qc_case_assignment
where qc_case_assignment.Cluster_0='2024-03-20';

INSERT INTO `p-cdh`.landing.qc_case_assignment
SELECT * FROM `d-cdh`.landing.qc_case_assignment_test_d;

In [0]:
%sql

CREATE TABLE `d-cdh`.landed.qc_case_assignment_test_d AS
SELECT DISTINCT *
FROM `p-cdh`.landed.qc_case_assignment
WHERE BusinessDate = '2024-03-19';

MERGE INTO `d-cdh`.landed.qc_case_assignment_test_d c
USING (
    SELECT case_code, assignment_process, extc_code
    FROM (
        SELECT row_number() OVER (
            PARTITION BY case_code 
            ORDER BY 
                CASE WHEN assignment_process = 'Unhandled' THEN 0 ELSE 1 END DESC, 
                CASE WHEN extc_code IS NULL THEN 0 ELSE 1 END DESC
        ) AS row_num,
        case_code, assignment_process, extc_code
        FROM `d-cdh`.landed.qc_case_assignment_test_d
        WHERE BusinessDate = '2024-03-19'
    ) AS a
    WHERE a.row_num = 2
) b
ON c.case_code = b.case_code 
   AND c.assignment_process = b.assignment_process 
   AND COALESCE(c.extc_code, 1) = COALESCE(b.extc_code, 1)
   AND c.BusinessDate = '2024-03-19'
WHEN MATCHED THEN
    DELETE;

DELETE from `p-cdh`.landed.qc_case_assignment
where qc_case_assignment.BusinessDate='2024-03-19';

INSERT INTO `p-cdh`.landed.qc_case_assignment
SELECT * FROM `d-cdh`.landed.qc_case_assignment_test_d;

## Refresh Views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)

